# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. By leveraging the Croissant schema and data model, we can flexibly query and analyze the dataset's content.

### Dataset Source
The dataset source is provided via a [Croissant schema](https://mlcommons.org/croissant/) URL.


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {getattr(metadata, 'version', None)}")

# Show collection timeframe
if hasattr(metadata, 'dataCollectionTimeframe'):
    print("Data collection period:", metadata.dataCollectionTimeframe)

## 2. Data Overview
Explore the available record sets (`@id`s), their fields and columns. For Croissant, record sets provide the main access to data tables in the dataset.

Let's enumerate the record sets defined in this Croissant schema and show their fields (all referenced by their `@id`).

In [ ]:
# List all available record sets, their fields, and columns.
from mlcroissant.models import RecordSet

record_sets = [obj for obj in metadata.objects if isinstance(obj, RecordSet)]
print(f"Number of record sets: {len(record_sets)}")
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    record_set_ids.append(rs.id)
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'No description'}")
    print(f"  Fields/Columns: ")
    for field in rs.fields:
        # Each field may have .id and .name
        print(f"    - {getattr(field, 'name', '')} (id: {field.id})")
    print("")
if not record_sets:
    print("No record sets defined in the metadata.")

## 3. Data Extraction
Load data from specific record set(s) using their `@id`. Here, we'll loop through all available record sets and convert each to a DataFrame.

Replace `<record_set_id>` below with the desired record set id you find above for more specific analysis.

In [ ]:
extracted_dataframes = dict()

# Use the discovered record_set_ids from above:
if record_set_ids:
    for recset_id in record_set_ids:
        recs = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(recs)
        extracted_dataframes[recset_id] = df
        print(f"--- RecordSet {recset_id} ---")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(), '\n')
else:
    print("No record sets to extract data from.")

## 4. Exploratory Data Analysis (EDA)
Let us perform EDA on a main record set.

First, select a record set with data (for example, demographic or survey responses), then operate on a numeric field (e.g., GAD-7 score, PHQ-9 score, or age).

All columns must be referenced using their Croissant `@id` for canonical access. Replace the values below if needed based on the actual column names for your dataset.

In [ ]:
# For demonstration: pick the first record set
if len(record_set_ids) > 0:
    main_record_set = record_set_ids[0]
    df = extracted_dataframes[main_record_set]
    print(f"Working with RecordSet: {main_record_set}, shape: {df.shape}")

    # Show candidate fields
    print("Available columns (with Croissant @id):")
    pprint.pprint(list(df.columns))

    # Example: Let's pick a likely numeric field (adjust as needed)
    # Try common score fields GAD-7, PHQ-9, PSQ, Age --
    score_candidates = [c for c in df.columns if any(x in c.lower() for x in ['gad', 'phq', 'psq', 'age', 'score'])]
    if score_candidates:
        # Just pick the first available numeric field
        numeric_field_id = score_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")
        # Remove NaN/convert to numeric
        col = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = col.mean()  # Use mean as an example threshold
        filtered_df = df[col > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
        print(filtered_df[[numeric_field_id]].head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (col - col.mean()) / col.std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records (first 5):")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a candidate category, e.g. 'gender', 'education', etc (based on @id)
        group_candidates = [c for c in df.columns if any(cc in c.lower() for cc in ['gender', 'village', 'education', 'category'])]
        if group_candidates:
            group_field_id = group_candidates[0]
            print(f"\nGrouping by: {group_field_id}")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped.head())
        else:
            print("No groupable categorical field available.")
    else:
        print("No obvious numeric field for EDA found.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Plot distributions or relationships for fields in the dataset.

We show how to visualize the distribution of a numeric field (if available) and its relationship with a selected categorical field.

_All visualizations use @id references for columns._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(record_set_ids) > 0 and score_candidates:
    # Distribution plot
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_candidates:
        # Boxplot by group variable
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=df[group_field_id], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric or groupable field available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated:
- How to load a FAIR and AI-ready Croissant-formatted dataset with `mlcroissant`.
- How to list record sets, fields, and access their `@id`s for flexible processing.
- How to load records into pandas DataFrames and perform exploratory filtering, normalization, grouping, and basic visualizations.

This approach can be adapted for any Croissant-compliant dataset by changing the schema URL and using the discovered `@id`s for entity reference throughout. For deeper analyses, explore additional fields, statistics or linkages as indicated by the schema.